In [18]:
import pandas as pd
import jenkspy
from pathlib import Path
import numpy as np

# ============================================================
# INPUT
# ============================================================
INPUT_CSV = Path("~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/data/district_final_risk_score.csv")

df = pd.read_csv(INPUT_CSV)

# ============================================================
# FILTER SUMMER 2026
# ============================================================

summer = df[df["timeperiod"].isin([
    "2025_04",
    "2025_05",
    "2025_06",
    "2025_07",
])].copy()

# ============================================================
# DISTRICT-WISE HEAT RISK
# (Change mean() to max() or sum() if desired)
# ============================================================

district_heat = (
    summer.groupby("district", as_index=False)["heat-days-score"]
    .mean()
)

# ============================================================
# NATURAL JENKS BREAKS
# # ============================================================

# N_CLASSES = 5

# breaks = jenkspy.jenks_breaks(
#     district_heat["heat-days-score"],
#     n_classes=N_CLASSES
# )

# district_heat["heat_risk_class"] = pd.cut(
#     district_heat["heat-days-score"],
#     bins=breaks,
#     labels=range(1, N_CLASSES + 1),
#     include_lowest=True
# )

# print(breaks)
# print(district_heat.head())

# ============================================================
# Z-SCORE BINNING
# ============================================================

mean = district_heat["heat-days-score"].mean()
std = district_heat["heat-days-score"].std()

district_heat["z_score"] = (
    district_heat["heat-days-score"] - mean
) / std

district_heat["heat_risk_class"] = pd.cut(
    district_heat["z_score"],
    bins=[-np.inf, -1, -0.5, 0.5, 1, np.inf],
    labels=[1, 2, 3, 4, 5],
    include_lowest=True
).astype(int)

print(f"Mean = {mean:.4f}")
print(f"Std Dev = {std:.4f}")

print(district_heat.head())


# ============================================================
# SAVE
# ============================================================
OUTPUT_CSV = Path("~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/2026_summer_district_hazard_risk_classes.csv")
district_heat.to_csv(OUTPUT_CSV, index=False)

print("Classification completed")
print(breaks)

print(f"\nSaved {len(district_heat)} districts to:")
print(OUTPUT_CSV.resolve())

Mean = 8.7631
Std Dev = 1.2143
    district  heat-days-score   z_score  heat_risk_class
0     Anugul           8.3650 -0.327836                3
1   Balangir          10.8450  1.714534                5
2  Baleshwar           6.1475 -2.154028                1
3    Bargarh           9.4500  0.565701                4
4    Bhadrak           7.0050 -1.447845                1
Classification completed
[np.float64(16.11), np.float64(16.93), np.float64(18.41333333333333), np.float64(19.6), np.float64(21.67), np.float64(23.916666666666668)]

Saved 30 districts to:
/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/2026_summer_district_hazard_risk_classes.csv


In [ ]:
# monthwise heat days score to compare with imd atlas

In [2]:
import pandas as pd
from pathlib import Path

# ============================================================
# INPUT
# ============================================================

INPUT_CSV = Path(
    "~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/data/district_final_risk_score.csv"
).expanduser()

df = pd.read_csv(INPUT_CSV)

# Extract month
df["month"] = df["timeperiod"].str[-2:]

# ============================================================
# APRIL
# ============================================================

april = (
    df[df["month"] == "04"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# MAY
# ============================================================

may = (
    df[df["month"] == "05"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# JUNE
# ============================================================

june = (
    df[df["month"] == "06"]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# ALL MONTHS (JAN-DEC)
# ============================================================

all_months = (
    df[df["month"].isin([f"{i:02d}" for i in range(1, 13)])]
    .groupby("district")["heat-days-score"]
    .sum()
)

# ============================================================
# COMBINE
# ============================================================

district_heat = pd.concat(
    [april, may, june, all_months],
    axis=1
).reset_index()

district_heat.columns = [
    "district",
    "April",
    "May",
    "June",
    "All_Months_Sum"
]

# ============================================================
# SAVE
# ============================================================

OUTPUT_CSV = Path(
    "~/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/district_heat_days_summary.csv"
).expanduser()

district_heat.to_csv(OUTPUT_CSV, index=False)

print(district_heat.head())
print(f"\nSaved {len(district_heat)} districts to:")
print(OUTPUT_CSV.resolve())

    district  April    May   June  All_Months_Sum
0     Anugul  93.48  61.66  89.85         1014.51
1   Balangir  75.28  59.70  67.27          949.72
2  Baleshwar  90.22  45.25  97.31          920.02
3    Bargarh  70.97  65.97  66.15          923.79
4    Bhadrak  95.91  43.46  92.52          942.06

Saved 30 districts to:
/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/RiskScoreModel/archive/analysis_scripts/district_heat_days_summary.csv
